# 3회차 실습: Matrix와 Matrix·Vector 곱 (Row·Column picture)

> Part 1: 3회차 (Matrix · Matrix·Vector 곱 · Row picture vs Column picture · Column space · Matrix·Matrix 곱 4해석)
> 사전 reading: MML §2.2, §2.7 (메인) / Strang Ch 1.3, 2.1 (발췌) / 3Blue1Brown EoLA Ch.3

이 노트북은 강의교안 3회차의 흐름(B 섹션 Matrix 정의 → C 섹션 Matrix·Vector 곱의 두 해석 → D 섹션 Column space → E 섹션 Matrix·Matrix 곱 4해석 → F 섹션 CS·AI 적용)을 그대로 따라갑니다. 매 절은 **Definition → (Theorem) → Application** 순서로 정리합니다.

## 학습 목표

이번 실습이 끝나면 다음을 NumPy 코드로 직접 보일 수 있습니다.

1. **Matrix**의 차원·Transpose·Symmetric 여부를 NumPy로 확인합니다.
2. **Matrix·Vector 곱** $A\mathbf{x}$를 **Row picture**(각 행과의 Dot product)와 **Column picture**(열들의 Linear combination)로 **직접 구현**하고, 두 결과가 `@`와 일치함을 `np.allclose`로 검증합니다.
3. **$A\mathbf{x} = \mathbf{b}$의 해 존재** $\iff$ $\mathbf{b} \in \mathrm{col}(A)$를 Rank로 판정합니다.
4. **Matrix·Matrix 곱**의 4가지 해석(원소·열·행·Outer product)으로 같은 곱 $AB$를 계산하고 모두 일치함을 보입니다.
5. **비가환성** $AB \ne BA$와 $(AB)^\top = B^\top A^\top$을 수치로 확인합니다.
6. **신경망 한 층** $\mathbf{y} = W\mathbf{x} + \mathbf{b}$, **그래프 Adjacency matrix**라는 두 적용을 코드로 체험합니다.

### 정의·정리될 객체 목록

| 번호 | 객체 |
|---|---|
| 정의 2.1 | $m \times n$ Matrix, 열 표현 $A = [\mathbf{a}_1 \mid \cdots \mid \mathbf{a}_n]$ |
| 정의 2.3 | Matrix·Vector 곱 $(A\mathbf{x})_i = \sum_j a_{ij} x_j$ |
| 정의 2.4 | Row picture 해석 (행과의 Dot product 묶음) |
| 정의 2.5 | Column picture 해석 (열들의 Linear combination) |
| 정의 2.6 | Matrix·Matrix 곱 $c_{ij} = \sum_l a_{il} b_{lj}$ |

### 사용 라이브러리

- NumPy, Matplotlib (필수)
- scikit-learn (mini-MNIST 한 층 통과 예시)

In [ ]:
# Colab 한글 폰트 설정 (matplotlib 깨짐 방지)
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # 한글 폰트 자동 등록 (NanumGothic)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
print('NumPy version:', np.__version__)

# 본 실습 내내 재사용할 Matrix와 Vector
A = np.array([[1, 2],
              [3, 4],
              [5, 6]], dtype=float)   # 3 x 2
x = np.array([2, -1], dtype=float)     # R^2
print('A =\n', A)
print('A.shape =', A.shape, '  (3행 2열)')
print('x =', x)

## 1. Definition: Matrix (정의 2.1·2.2)

### 정의 2.1 ($m \times n$ Matrix)
$m$행 $n$열의 실수 표 $A = (a_{ij}) \in \mathbb{R}^{m \times n}$. $i$는 행 인덱스, $j$는 열 인덱스입니다.

### 정의 2.2 (열 표현)
$A$의 $j$번째 열을 $\mathbf{a}_j \in \mathbb{R}^m$이라 하면 $A = [\mathbf{a}_1 \mid \mathbf{a}_2 \mid \cdots \mid \mathbf{a}_n]$.

### Transpose·Symmetric
$(A^\top)_{ij} = a_{ji}$. 정사각 Matrix가 $A^\top = A$이면 **Symmetric(대칭)**입니다.

In [ ]:
# 차원, Transpose, 각 열
print('차원 (m, n) =', A.shape)
print('A^T =\n', A.T, '  shape', A.T.shape)
print('1번째 열 a_1 =', A[:, 0])
print('2번째 열 a_2 =', A[:, 1])

# Symmetric 판정: A^T == A 여야 하고, 정사각이어야 한다
def is_symmetric(M):
    M = np.asarray(M, dtype=float)
    return M.shape[0] == M.shape[1] and np.allclose(M, M.T)

S = np.array([[1, 2, 0],
              [2, 3, 4],
              [0, 4, 5]], dtype=float)
print('\nS =\n', S)
print('S는 Symmetric?', is_symmetric(S))
print('A는 Symmetric?', is_symmetric(A), '(정사각이 아니므로 정의되지 않음 -> False)')

## 2. Definition: Matrix·Vector 곱 (정의 2.3)

$A \in \mathbb{R}^{m \times n}$, $\mathbf{x} \in \mathbb{R}^n$에 대해 $A\mathbf{x} \in \mathbb{R}^m$의 $i$성분:
$$(A\mathbf{x})_i = \sum_{j=1}^{n} a_{ij}\, x_j$$

조건은 $A$의 **열 수** $=$ $\mathbf{x}$의 차원. 먼저 정의 그대로 이중 for문으로 구현하고 NumPy `@`와 비교합니다.

In [ ]:
def matvec_naive(A, x):
    '''정의 그대로: (Ax)_i = sum_j a_ij x_j (이중 for문).'''
    A = np.asarray(A, dtype=float); x = np.asarray(x, dtype=float)
    m, n = A.shape
    assert x.shape[0] == n, '열 수와 벡터 차원 불일치'
    out = np.zeros(m)
    for i in range(m):
        for j in range(n):
            out[i] += A[i, j] * x[j]
    return out

print('matvec_naive(A, x) =', matvec_naive(A, x))
print('A @ x             =', A @ x)
print('일치?', np.allclose(matvec_naive(A, x), A @ x))

## 3. 두 해석: Row picture vs Column picture (정의 2.4·2.5)

같은 정의를 **두 가지로 묶어** 봅니다.

- **Row picture (정의 2.4)**: $A\mathbf{x}$의 $i$성분 $= \mathbf{r}_i \cdot \mathbf{x}$ (행과 $\mathbf{x}$의 Dot product).
- **Column picture (정의 2.5)**: $A\mathbf{x} = x_1 \mathbf{a}_1 + \cdots + x_n \mathbf{a}_n$ (열들의 Linear combination).

세 결과(`@` 포함)가 모두 같아야 합니다.

In [ ]:
def matvec_row(A, x):
    '''Row picture: 각 행과 x의 Dot product.'''
    A = np.asarray(A, dtype=float); x = np.asarray(x, dtype=float)
    return np.array([np.dot(A[i, :], x) for i in range(A.shape[0])])

def matvec_col(A, x):
    '''Column picture: 열들의 Linear combination.'''
    A = np.asarray(A, dtype=float); x = np.asarray(x, dtype=float)
    out = np.zeros(A.shape[0])
    for j in range(A.shape[1]):
        out += x[j] * A[:, j]
    return out

r = matvec_row(A, x)
c = matvec_col(A, x)
print('Row picture   :', r)
print('Column picture:', c)
print('NumPy A @ x   :', A @ x)
assert np.allclose(r, c) and np.allclose(r, A @ x)
print('\n세 방법 모두 일치 (OK)')

# 손계산 확인용 분해 출력
print('\n[Row]   1행 dot x =', np.dot(A[0], x), ', 2행 =', np.dot(A[1], x), ', 3행 =', np.dot(A[2], x))
print('[Col]  ', x[0], '* a_1 +', x[1], '* a_2 =', x[0]*A[:,0], '+', x[1]*A[:,1], '=', c)

## 4. 시각화: 두 해석을 한 그림으로

연립방정식 $\;x_1 + 2x_2 = 4,\;\; 3x_1 + 2x_2 = 6\;$ (해 $(1,\ 1.5)$)을 두 그림으로 봅니다.

- **Row picture**: 각 식이 직선 한 개, 해는 두 직선의 교점.
- **Column picture**: 두 열 $\mathbf{a}_1, \mathbf{a}_2$를 $x_1, x_2$배 섞어 $\mathbf{b}$에 도달.

In [ ]:
M = np.array([[1.0, 2.0], [3.0, 2.0]])   # 계수 행렬
bvec = np.array([4.0, 6.0])
sol = np.linalg.solve(M, bvec)
print('해 (x1, x2) =', sol)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.4))

# Row picture
xs = np.linspace(-1, 5, 200)
axL.plot(xs, (4 - xs) / 2, label=r'$x_1 + 2x_2 = 4$')
axL.plot(xs, (6 - 3 * xs) / 2, label=r'$3x_1 + 2x_2 = 6$')
axL.plot(*sol, 'ko', ms=9)
axL.annotate(f'({sol[0]:.0f}, {sol[1]:.1f})', sol, textcoords='offset points', xytext=(10, 8))
axL.axhline(0, color='gray', lw=0.7); axL.axvline(0, color='gray', lw=0.7)
axL.set_xlim(-1, 5); axL.set_ylim(-1, 4); axL.legend(); axL.grid(alpha=0.3)
axL.set_title('Row picture: 직선들의 교점'); axL.set_xlabel('$x_1$'); axL.set_ylabel('$x_2$')

# Column picture
a1 = M[:, 0]; a2 = M[:, 1]
def arr(ax, s, v, **kw): ax.annotate('', xy=s + v, xytext=s, arrowprops=dict(arrowstyle='-|>', lw=2, **kw))
O = np.zeros(2)
arr(axR, O, a1, color='tab:blue'); arr(axR, O, a2, color='tab:red')
arr(axR, O, sol[0]*a1, color='tab:blue', linestyle='dashed')
arr(axR, sol[0]*a1, sol[1]*a2, color='tab:red', linestyle='dashed')
arr(axR, O, bvec, color='tab:green')
axR.text(*(a1*0.5), r'$\mathbf{a}_1$', color='tab:blue')
axR.text(*(a2*0.5), r'$\mathbf{a}_2$', color='tab:red')
axR.text(*(bvec*0.55), r'$\mathbf{b}$', color='tab:green')
axR.axhline(0, color='gray', lw=0.7); axR.axvline(0, color='gray', lw=0.7)
axR.set_xlim(-0.5, 5.5); axR.set_ylim(-0.5, 7); axR.grid(alpha=0.3)
axR.set_title(r'Column picture: $1\,\mathbf{a}_1 + 1.5\,\mathbf{a}_2 = \mathbf{b}$')
fig.suptitle('같은 해, 두 가지 해석', fontweight='bold')
fig.tight_layout(); plt.show()

## 5. Theorem/Application: $A\mathbf{x} = \mathbf{b}$와 Column space

**$A\mathbf{x} = \mathbf{b}$의 해가 존재** $\iff$ $\mathbf{b}$가 $A$의 열들의 Linear combination으로 표현됨 $\iff$ $\mathbf{b} \in \mathrm{col}(A)$.

판정법: $\mathrm{rank}(A) = \mathrm{rank}([A \mid \mathbf{b}])$이면 해가 존재합니다 (Rouché-Capelli).

In [ ]:
# 두 열이 평행한 Matrix: col(A)는 R^3의 직선 (1차원)
A2 = np.array([[1.0, 2.0],
               [2.0, 4.0],
               [3.0, 6.0]])
print('rank(A2) =', np.linalg.matrix_rank(A2), ' -> col(A2)는 1차원 직선')

def solvable(A, b):
    A = np.asarray(A, float); b = np.asarray(b, float).reshape(-1, 1)
    aug = np.hstack([A, b])
    return np.linalg.matrix_rank(A) == np.linalg.matrix_rank(aug)

b_in  = np.array([1.0, 2.0, 3.0])   # = 1 * (1,2,3) -> 직선 위
b_out = np.array([1.0, 1.0, 1.0])   # 직선 밖
for b in (b_in, b_out):
    print(f'b = {b}:  해 존재? {solvable(A2, b)}')

## 6. 시각화: Column space (직선 vs 평면)

열이 **종속**이면 $\mathrm{col}(A)$가 직선으로 무너지고, **독립**이면 평면을 채웁니다.

In [ ]:
fig = plt.figure(figsize=(11, 4.6))

# 종속: 직선
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
a = np.array([1.0, 2.0, -1.0]); ts = np.linspace(-2, 2, 40)
ln = np.outer(ts, a)
ax1.plot(ln[:, 0], ln[:, 1], ln[:, 2], color='tab:blue')
ax1.quiver(0, 0, 0, *a, color='tab:red')
ax1.set_title(r'열 종속: col(A) = 직선')

# 독립: 평면
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
v1 = np.array([1.0, 0, 1]); v2 = np.array([0, 1.0, 1])
ss, tt = np.meshgrid(np.linspace(-2, 2, 10), np.linspace(-2, 2, 10))
P = ss[..., None] * v1 + tt[..., None] * v2
ax2.plot_surface(P[..., 0], P[..., 1], P[..., 2], alpha=0.35, color='tab:green')
ax2.quiver(0, 0, 0, *v1, color='tab:blue'); ax2.quiver(0, 0, 0, *v2, color='tab:red')
ax2.set_title(r'열 독립: col(A) = 평면')
fig.tight_layout(); plt.show()

## 7. Definition: Matrix·Matrix 곱의 4가지 해석 (정의 2.6)

$C = AB$, $c_{ij} = \sum_l a_{il} b_{lj}$를 4가지로 계산합니다.

1. **원소**: $c_{ij} = (A$의 $i$행$) \cdot (B$의 $j$열$)$
2. **열**: $AB = [A\mathbf{b}_1 \mid \cdots \mid A\mathbf{b}_n]$
3. **행**: $AB$의 $i$행 $= (A$의 $i$행$)\,B$
4. **Outer product 합**: $AB = \sum_l \mathbf{a}_l \mathbf{b}_l^\top$

In [ ]:
P = np.array([[1.0, 2.0], [3.0, 4.0]])
Q = np.array([[5.0, 6.0], [7.0, 8.0]])
ref = P @ Q

# 1. 원소
C1 = np.array([[np.dot(P[i], Q[:, j]) for j in range(2)] for i in range(2)])
# 2. 열
C2 = np.column_stack([P @ Q[:, j] for j in range(Q.shape[1])])
# 3. 행
C3 = np.vstack([P[i, :] @ Q for i in range(P.shape[0])])
# 4. Outer product 합
C4 = sum(np.outer(P[:, l], Q[l, :]) for l in range(P.shape[1]))

for name, C in [('원소', C1), ('열', C2), ('행', C3), ('Outer합', C4)]:
    print(f'{name:7s}일치? {np.allclose(C, ref)}')
print('\nAB =\n', ref)

## 8. Theorem: 비가환성 $AB \ne BA$ 와 $(AB)^\top = B^\top A^\top$

Matrix 곱은 함수 합성이므로 일반적으로 **비가환**입니다. 또한 Transpose는 곱의 순서를 뒤집습니다.

In [ ]:
U = np.array([[0.0, 1.0], [0.0, 0.0]])
V = np.array([[0.0, 0.0], [1.0, 0.0]])
print('UV =\n', U @ V)
print('VU =\n', V @ U)
print('UV == VU ?', np.allclose(U @ V, V @ U))

print('\n(PQ)^T == Q^T P^T ?', np.allclose((P @ Q).T, Q.T @ P.T))

## 9. Application: 신경망 한 층 = $W\mathbf{x} + \mathbf{b}$ (mini-MNIST)

8×8 손글씨 숫자 한 장을 $\mathbb{R}^{64}$ Vector로 보고, 가중치 $W \in \mathbb{R}^{10 \times 64}$를 곱해 10개 클래스 점수를 얻습니다. 이것이 **선형 분류기 한 층** $\mathbf{y} = W\mathbf{x} + \mathbf{b}$입니다.

In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()
img = digits.images[0]            # 8x8
xvec = img.reshape(-1)            # R^64
print('이미지 -> Vector 차원:', xvec.shape)

rng = np.random.default_rng(3)
W = rng.standard_normal((10, 64)) * 0.1   # 10 클래스 x 64 픽셀
bias = np.zeros(10)
y = W @ xvec + bias               # R^10 점수
print('한 층 출력 y =', np.round(y, 2))
print('argmax (예측 클래스) =', int(np.argmax(y)))

# Row picture: 각 출력 = 한 뉴런(가중치 행)과 입력의 Dot product
print('\nRow picture 확인: y[0] == W[0] . xvec ?', np.allclose(y[0], np.dot(W[0], xvec)))
# Column picture: y = sum_j xvec[j] * W[:, j]
y_col = sum(xvec[j] * W[:, j] for j in range(64))
print('Column picture 확인:', np.allclose(y_col, y))

## 10. Application: 그래프 Adjacency matrix

노드 $i \to j$ 간선이 있으면 $a_{ij} = 1$인 Matrix가 **Adjacency matrix(인접행렬)**입니다.
$A^2$의 $(i, j)$는 $i$에서 $j$로 가는 **길이 2 경로의 수**입니다 (Matrix 곱의 의미).

In [ ]:
# 4개 노드 방향 그래프: 1->2, 2->3, 3->4, 4->1, 1->3
Adj = np.zeros((4, 4))
for (i, j) in [(0, 1), (1, 2), (2, 3), (3, 0), (0, 2)]:
    Adj[i, j] = 1
print('Adjacency matrix A =\n', Adj.astype(int))

# 각 노드의 out-degree = A @ ones
print('out-degree (A @ 1) =', (Adj @ np.ones(4)).astype(int))

# 길이 2 경로 수
print('A^2 (길이 2 경로 수) =\n', (Adj @ Adj).astype(int))

# heatmap 시각화
fig, ax = plt.subplots(figsize=(4.2, 3.6))
im = ax.imshow(Adj, cmap='Blues', vmin=0, vmax=1)
for i in range(4):
    for j in range(4):
        ax.text(j, i, int(Adj[i, j]), ha='center', va='center',
                color='white' if Adj[i, j] else '#444')
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(range(1, 5)); ax.set_yticklabels(range(1, 5))
ax.set_xlabel('도착 노드 j'); ax.set_ylabel('출발 노드 i'); ax.set_title('Adjacency matrix')
plt.show()

## 11. 연습 (자가 점검)

아래를 직접 코드로 풀어 보세요.

1. $A = \begin{pmatrix} 1 & 0 & 1 \\ 0 & 1 & 1 \\ 1 & 1 & 2 \end{pmatrix}$, $\mathbf{x} = (1,1,1)^\top$에 대해 `matvec_row`, `matvec_col`, `A @ x`가 모두 일치함을 보이시오.
2. 위 $A$의 셋째 열이 처음 두 열의 합임을 확인하고, $\mathrm{rank}(A)$를 구하시오. $\mathrm{col}(A)$는 직선·평면·전체 중 무엇인가?
3. $A\mathbf{y} = (1, 2, 3)^\top$의 해 존재 여부를 `solvable`로 판정하시오.
4. 임의의 $3 \times 3$ Matrix 두 개로 $(AB)^\top = B^\top A^\top$과 $AB \ne BA$를 확인하시오.

## 12. 정리

오늘 도입한 정의 목록:

| 번호 | 내용 |
|---|---|
| 정의 2.1 | $m \times n$ Matrix, 열 표현 $A = [\mathbf{a}_1 \mid \cdots \mid \mathbf{a}_n]$ |
| 정의 2.3 | Matrix·Vector 곱 $(A\mathbf{x})_i = \sum_j a_{ij} x_j$ |
| 정의 2.4 | Row picture: $A\mathbf{x}$의 $i$성분 $= \mathbf{r}_i \cdot \mathbf{x}$ |
| 정의 2.5 | Column picture: $A\mathbf{x} = \sum_j x_j \mathbf{a}_j$ |
| 정의 2.6 | Matrix·Matrix 곱 $c_{ij} = \sum_l a_{il} b_{lj}$ |

오늘 검증한 사실:

- $A\mathbf{x}$의 **Row picture·Column picture는 같은 결과**이며 `@`와 일치한다.
- $A\mathbf{x} = \mathbf{b}$의 **해 존재 $\iff$ $\mathbf{b} \in \mathrm{col}(A)$**이고, Rank로 판정한다.
- Matrix·Matrix 곱은 **원소·열·행·Outer product** 4가지로 계산해도 모두 같다.
- $AB \ne BA$ (비가환), $(AB)^\top = B^\top A^\top$ (순서 뒤집힘).
- **신경망 한 층** $W\mathbf{x}$는 Column picture(입력 특징의 가중 혼합)·Row picture(뉴런별 가중합) 두 관점으로 읽힌다.
- **Adjacency matrix**에서 $A^2$는 길이 2 경로의 수를 센다.

### 다음 회차(4회차)와의 연결

오늘 본 $A\mathbf{x} = \mathbf{b}$의 **해 존재**를 4회차에서 **가우스 소거법**으로 구체적으로 풉니다. 소거 과정은 본 회차 Row picture의 체계적 운영이며, 그 결과인 RREF가 Column space와 해의 구조를 직접 드러냅니다.